# ABSA Hotel Reviews: English vs Spanish Models

**Authors:** Darragh Kerins and Ander Peña

This notebook compares 4 models for Aspect-Based Sentiment Analysis on hotel reviews:
- **Model A:** English baseline (real M-ABSA data)
- **Model B:** Spanish (machine-translated)
- **Model C:** Spanish (synthetic LLM data)
- **Model D:** Spanish (combined translated + synthetic)

Research question: Does synthetic Spanish outperform machine-translated Spanish?

## Cell 1: Install Required Libraries

Installs transformers (for mT5), datasets (for M-ABSA), and accelerate (for GPU training).

In [ ]:
!pip install transformers datasets accelerate -q

## Cell 2: Mount Google Drive

Mount Google Drive to access project files.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cell 3: Import Libraries

Import all libraries and detect GPU/CPU device.

In [ ]:
import torch
import numpy as np
import os
import glob
from transformers import MT5ForConditionalGeneration, AutoTokenizer, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq, AutoConfig
from datasets import Dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Cell 4: Helper Functions

Functions for data processing and loading.

In [ ]:
def parse_example(text):
    # Parse review####[[triplets]] into (review, triplets)
    if "####" not in text:
        return text, ""
    review, labels = text.split("####", 1)
    try:
        triplets = eval(labels)
        formatted = "; ".join([f"({t[0]}, {t[1]}, {t[2]})" for t in triplets])
    except:
        formatted = ""
    return review.strip(), formatted

def parse_prediction(text):
    # Parse model output into triplet tuples
    triplets = []
    for part in text.split(";"):
        part = part.strip()
        if part.startswith("(") and part.endswith(")"):
            pieces = [p.strip() for p in part[1:-1].split(",")]
            if len(pieces) == 3:
                triplets.append((pieces[0], pieces[1], pieces[2]))
    return triplets

def tokenize_data(examples, tokenizer):
    # Convert examples to tokenized model inputs
    inputs, targets = [], []
    for text in examples["text"]:
        review, triplets = parse_example(text)
        inputs.append(f"review: {review}")
        targets.append(triplets)
    model_inputs = tokenizer(inputs, max_length=192, padding="max_length", truncation=True)
    labels = tokenizer(targets, max_length=96, padding="max_length", truncation=True)
    pad_id = tokenizer.pad_token_id
    model_inputs["labels"] = [[-100 if t == pad_id else t for t in seq] for seq in labels["input_ids"]]
    return model_inputs

def load_data_from_file(filepath):
    # Load data from text file in M-ABSA format
    examples = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line and "####" in line:
                examples.append({"text": line})
    return Dataset.from_list(examples)

## Cell 5: Load English Hotel Data

Load English data from local Data/Data/ files.

In [ ]:
print("Loading English hotel data from Data/Data/...")
en_train = load_data_from_file("/content/drive/MyDrive/Colab Notebooks/Project/Data/en_train.txt")
en_test = load_data_from_file("/content/drive/MyDrive/Colab Notebooks/Project/Data/en_test.txt")
print(f"English: {len(en_train)} train, {len(en_test)} test")

## Cell 6: Load Spanish (Machine-Translated) Data

Load Spanish data from local Data/Data/ files.

In [ ]:
print("Loading Spanish (machine-translated) from Data/Data/...")
es_trans_train = load_data_from_file("/content/drive/MyDrive/Colab Notebooks/Project/Data/es_train.txt")
es_trans_test = load_data_from_file("/content/drive/MyDrive/Colab Notebooks/Project/Data/es_test.txt")
print(f"Spanish (translated): {len(es_trans_train)} train, {len(es_trans_test)} test")

## Cell 7: Load Synthetic Spanish Data

Load LLM-generated synthetic Spanish data from Data/Data/ folder.

In [ ]:
print("Loading synthetic Spanish data from Data/Data/...")
synthetic_examples = []
for filepath in glob.glob("/content/drive/MyDrive/Colab Notebooks/Project/Data_synthetic/synthetic_train_all.txt"):
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line and "####" in line:
                synthetic_examples.append({"text": line})
split_idx = int(0.9 * len(synthetic_examples))
es_syn_train = Dataset.from_list(synthetic_examples[:split_idx])
es_syn_test = Dataset.from_list(synthetic_examples[split_idx:])
print(f"Synthetic: {len(es_syn_train)} train, {len(es_syn_test)} test")

## Cell 8: Tokenize All Datasets

Initialize mT5 tokenizer and convert all datasets to model format.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("google/mt5-small")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

en_train_proc = en_train.map(lambda x: tokenize_data(x, tokenizer), batched=True, remove_columns=en_train.column_names)
en_test_proc = en_test.map(lambda x: tokenize_data(x, tokenizer), batched=True, remove_columns=en_test.column_names)
es_trans_train_proc = es_trans_train.map(lambda x: tokenize_data(x, tokenizer), batched=True, remove_columns=es_trans_train.column_names)
es_trans_test_proc = es_trans_test.map(lambda x: tokenize_data(x, tokenizer), batched=True, remove_columns=es_trans_test.column_names)
es_syn_train_proc = es_syn_train.map(lambda x: tokenize_data(x, tokenizer), batched=True, remove_columns=es_syn_train.column_names)
es_syn_test_proc = es_syn_test.map(lambda x: tokenize_data(x, tokenizer), batched=True, remove_columns=es_syn_test.column_names)
print("All datasets tokenized!")

## Cell 9: Define Training Function

Helper function to train mT5 model.

**Note:** fp16 disabled to prevent NaN validation loss. bf16 used if available.

In [ ]:
def train_model(train_data, test_data, output_dir, epochs=3):
    config = AutoConfig.from_pretrained("google/mt5-small")
    config.tie_word_embeddings = False
    model = MT5ForConditionalGeneration.from_pretrained("google/mt5-small", config=config)
    model = model.to(device)
    args = Seq2SeqTrainingArguments(
        output_dir=output_dir,
        num_train_epochs=epochs,
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        learning_rate=3e-4,
        logging_steps=50,
        save_strategy="epoch",
        eval_strategy="epoch",
        fp16=False,
        bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
        predict_with_generate=True,
        generation_max_length=96,
        report_to=[]
    )
    data_collator = DataCollatorForSeq2Seq(
        tokenizer, model=model,
        padding=True, return_tensors="pt"
    )
    trainer = Seq2SeqTrainer(
        model=model, args=args,
        train_dataset=train_data, eval_dataset=test_data,
        data_collator=data_collator,
        processing_class=tokenizer
    )
    trainer.train()
    trainer.save_model()
    tokenizer.save_pretrained(output_dir)
    return model

## Cell 10: Train Model A - English Baseline

Train on real English M-ABSA data.

In [ ]:
print("Training Model A: English Baseline...")
model_a = train_model(en_train_proc, en_test_proc,
    "/content/drive/MyDrive/Colab Notebooks/Project/checkpoints/model_a_en", epochs=5)


## Cell 11: Train Model B - Spanish (Translated)

Train on machine-translated Spanish data.

In [ ]:

print("Training Model B: Spanish (Translated)...")
model_b = train_model(es_trans_train_proc, es_trans_test_proc,
    "/content/drive/MyDrive/Colab Notebooks/Project/checkpoints/model_b_es_trans", epochs=5)


## Cell 12: Train Model C - Spanish (Synthetic)

Train on LLM-generated synthetic Spanish data.

In [ ]:
print("Training Model C: Spanish (Synthetic)...")
model_c = train_model(es_syn_train_proc, es_syn_test_proc,
    "/content/drive/MyDrive/Colab Notebooks/Project/checkpoints/model_c_es_syn", epochs=5)


## Cell 13: Train Model D - Spanish (Combined)

Train on combined translated + synthetic data.

In [ ]:
print("Training Model D: Spanish (Combined)...")
es_combined_train = Dataset.from_list(list(es_trans_train_proc) + list(es_syn_train_proc))
model_d = train_model(es_combined_train, es_trans_test_proc,
    "/content/drive/MyDrive/Colab Notebooks/Project/checkpoints/model_d_es_combined", epochs=5)

## Cell 14: Define Evaluation Function

Calculate precision, recall, and F1 for triplet extraction.

In [ ]:
def evaluate(model, test_data, label):
    model.eval()
    preds, gold = [], []
    for i in range(len(test_data)):
        text = test_data[i]["text"]
        _, true = parse_example(text)
        gold.append(parse_prediction(true))
        review, _ = parse_example(text)
        inputs = tokenizer(f"review: {review}", return_tensors="pt", max_length=192, truncation=True).to(device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=96, num_beams=4)
        pred_text = tokenizer.decode(out[0], skip_special_tokens=True)
        preds.append(parse_prediction(pred_text))
    tp = fp = fn = 0
    for g, p in zip(gold, preds):
        g_set, p_set = set(g), set(p)
        tp += len(g_set & p_set)
        fp += len(p_set - g_set)
        fn += len(g_set - p_set)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    print(f"{label}: P={precision:.2%}, R={recall:.2%}, F1={f1:.2%}")
    return f1

## Cell 15: Evaluate All Models

Test all 4 models including cross-lingual (English model on Spanish data).

In [ ]:
results = {}
print("\n=== MODEL A: English Baseline ===")
results["A_EN_on_EN"] = evaluate(model_a, en_test, "EN on EN")
results["A_EN_on_ES"] = evaluate(model_a, es_trans_test, "EN on ES (cross-lingual)")
print("\n=== MODEL B: Spanish (Translated) ===")
results["B_ES_trans_on_ES"] = evaluate(model_b, es_trans_test, "ES-trans on ES-trans")
print("\n=== MODEL C: Spanish (Synthetic) ===")
results["C_ES_syn_on_ES"] = evaluate(model_c, es_trans_test, "ES-syn on ES-trans")
print("\n=== MODEL D: Spanish (Combined) ===")
results["D_ES_combined_on_ES"] = evaluate(model_d, es_trans_test, "ES-combined on ES-trans")

## Cell 16: Results Comparison Table

Shows the key finding: does synthetic beat translated?

In [ ]:
print("\n=== RESULTS TABLE ===")
print("Model | Training Data   | Test Data  | F1")
print("-" * 50)
print(f"A     | EN real         | EN real    | {results.get('A_EN_on_EN', 0):.3f}")
print(f"A     | EN real         | ES trans   | {results.get('A_EN_on_ES', 0):.3f} (cross-lingual)")
print(f"B     | ES translated   | ES trans   | {results.get('B_ES_trans_on_ES', 0):.3f}")
print(f"C     | ES synthetic    | ES trans   | {results.get('C_ES_syn_on_ES', 0):.3f}")
print(f"D     | ES combined     | ES trans   | {results.get('D_ES_combined_on_ES', 0):.3f}")
print("\nKey Finding:")
b_f1 = results.get('B_ES_trans_on_ES', 0)
c_f1 = results.get('C_ES_syn_on_ES', 0)
if c_f1 > b_f1:
    print(f"Synthetic beats translated by {c_f1 - b_f1:.3f} F1")
else:
    print(f"Translated beats synthetic by {b_f1 - c_f1:.3f} F1")

## Cell 17: Test Predictions

Compare all 4 models on sample English and Spanish reviews.

In [ ]:
def predict(model, text):
    model.eval()
    inputs = tokenizer(f"review: {text}", return_tensors="pt", max_length=192, truncation=True).to(device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=96, num_beams=4)
    return parse_prediction(tokenizer.decode(out[0], skip_special_tokens=True))

tests = [
    ("The bathroom was disgusting but staff were helpful.", "EN"),
    ("El baño era asqueroso pero el personal fue servicial.", "ES")
]
for review, lang in tests:
    print(f"\n{lang}: {review}")
    print(f"  A (EN):       {predict(model_a, review)}")
    print(f"  B (ES-trans): {predict(model_b, review)}")
    print(f"  C (ES-syn):   {predict(model_c, review)}")
    print(f"  D (ES-combo): {predict(model_d, review)}")